In [62]:
from copy import deepcopy
import torch

import numpy as np
import pandas as pd
import cv2
import os
import os.path as osp
from pathlib import Path

from tqdm import tqdm
from utils.ioutils import load_pickle, load_json
from utils.process_video import video_results_to_submission_format
from utils.driver_state import determine_drive_state_changed

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [63]:
ALLOWED_CLASSES = [
    # "person",
    # "bicycle",
    # "car",
    # "motorcycle",
    # "bus",
    # "truck",
]

METADATA_PATH = "/media/marek/disk/datasets/sifco-accident/labels.csv"
annotations = pd.read_csv(METADATA_PATH)
ANNOTATIONS_DIR = "./inference-yolo11x"

annotation_paths = []

for i, row in annotations.iterrows():
    _video_path = row["path"]
    detections_path = os.path.join(ANNOTATIONS_DIR, osp.dirname(_video_path), Path(_video_path).stem + ".json")
    assert osp.exists(detections_path), detections_path
    annotation_paths.append(detections_path)

annotations["detections_path"] = annotation_paths

## Bounding box area dynamic

In [64]:
from utils.process_video import process_detections

MAX_SECS = 10
hazard_predictions = []
for i, row in tqdm(annotations.iterrows(), total=len(annotations)):
    fps = row["no_frames"] / row["duration"]

    detections_path = row["detections_path"]
    detections = load_json(detections_path)

    detections = [det for det in detections if det["frame_number"] / fps <= MAX_SECS]

    hazards = process_detections(detections, allowed_classes=ALLOWED_CLASSES)
    hazard_predictions.append(hazards)

100%|██████████| 2027/2027 [00:23<00:00, 86.18it/s] 


In [65]:
def temporal_accuracy(x: list[int], y: list[int], sigmas: list[float]) -> float:
    x, y = np.array(x), np.array(y)
    sigmas = np.array(sigmas)
    scores = np.exp( -(x - y)**2 / (2 * sigmas**2))
    score = np.mean(scores)
    return score


x_bbox_dynamic = [v["bbox_sizes_frame"] for v in hazard_predictions]
y = annotations["accident_frame"].values
y_total_frames, y_durations_sec = annotations["no_frames"].values, annotations["duration"].values
y_fps = y_total_frames / y_durations_sec

print("Bounding box area dynamic")
for weight in [1/2, 1, 2]:
    sigmas = y_fps * weight
    acc = temporal_accuracy(x_bbox_dynamic, y, sigmas)
    print(f" - Weight: {weight:.2f}: {acc:.3f}")


Bounding box area dynamic
 - Weight: 0.50: 0.132
 - Weight: 1.00: 0.259
 - Weight: 2.00: 0.462


## Optical flow

In [66]:
optical_flow_path = "./optical-flow/optical_flow.pkl"
optical_flow_data = torch.load(
    optical_flow_path, weights_only=False
)
records = []

# Iterate over the optical flow data to prepare a list of records
for video_filename, frames in optical_flow_data.items():
    for frame_info in frames:
        records.append(
            {
                "filename": video_filename,
                "frame": frame_info["frame"],
                "score": frame_info["score"],
            }
        )

optical_flow_df = pd.DataFrame(records)
optical_flow_df["video"] = optical_flow_df["filename"].apply(
    lambda filename: osp.basename(filename)
)

video_groups = []
for video_id, video_group in optical_flow_df.groupby("video"):
    # Backfill the missing score values
    time_series_scores = video_group["score"].bfill().reset_index(drop=True)

    # Determine the driver state change based on the backfilled scores
    video_group["change_bkp4"] = determine_drive_state_changed(
        time_series_scores, n_bkps=4
    )

    video_groups.append(video_group)

final_optical_flow_df = pd.concat(video_groups).reset_index(drop=True)

In [67]:
# video_name -> frame of change
prediction_mapping = {}
# file_names = final_optical_flow_df["filename"].tolist()
# for file_name in tqdm(file_names, total=len(file_names)):
#     predictions = final_optical_flow_df[final_optical_flow_df["filename"] == file_name]["change_bkp4"]
#     detected_hazard_frame = next((i for i, val in enumerate(predictions) if val == True), None)
#     prediction_mapping[file_name] = detected_hazard_frame

prediction_mapping = (
    final_optical_flow_df[final_optical_flow_df["change_bkp4"]]
    .groupby("filename")["frame"]
    .min()
    .to_dict()
)

print(len(prediction_mapping))


2019


In [68]:
x_optical_flow = []
for i, row in tqdm(annotations.iterrows(), total=len(annotations)):
    video_path = row["path"]
    video_file = osp.basename(video_path)
    predicted_frame = prediction_mapping.get(video_file, None)
    if predicted_frame is not None:
        x_optical_flow.append(predicted_frame)
    else:
        x_optical_flow.append(0)
        print(f"{video_file} is not in optical flow")



  0%|          | 0/2027 [00:00<?, ?it/s]

-i9bRJWMtTo_00.mp4 is not in optical flow
WTz4apKhaQY_00.mp4 is not in optical flow
IcHtsAFIRjk_00.mp4 is not in optical flow
m8HHbAqyx8g_00.mp4 is not in optical flow
l6WnoOecA9s_00.mp4 is not in optical flow
mhLc2Vl1ekk_00.mp4 is not in optical flow
Sq5i3O5yoAY_00.mp4 is not in optical flow
651RQwxB3WA_1_00.mp4 is not in optical flow


100%|██████████| 2027/2027 [00:00<00:00, 30240.21it/s]


In [69]:
y = annotations["accident_frame"].values
y_total_frames, y_durations_sec = annotations["no_frames"].values, annotations["duration"].values
y_fps = y_total_frames / y_durations_sec


print("Optical flow dynamic")
for weight in [1/2, 1, 2]:
    sigmas = np.ones_like(x_optical_flow) * weight
    acc = temporal_accuracy(x_optical_flow / y_fps, y / y_fps, sigmas)
    print(f" - Weight: {weight:.2f}: {acc:.3f}")

Optical flow dynamic
 - Weight: 0.50: 0.118
 - Weight: 1.00: 0.236
 - Weight: 2.00: 0.433


In [70]:
x_ensemble = np.mean([x_bbox_dynamic, x_optical_flow], axis=0).round()

print("BBox + OF - Average")
for weight in [1/2, 1, 2]:
    sigmas = y_fps * weight
    acc = temporal_accuracy(x_ensemble, y, sigmas)
    print(f" - Weight: {weight:.2f}: {acc:.3f}")


BBox + OF - Average
 - Weight: 0.50: 0.143
 - Weight: 1.00: 0.280
 - Weight: 2.00: 0.489


In [71]:
# ensemble_frame_metadata = {}
# for i, row in tqdm(annotations.iterrows(), total=len(annotations)):
#     file_name = row["path"]
#     frame = x_ensemble[i]
#     ensemble_frame_metadata[file_name] = frame
#
# torch.save(ensemble_frame_metadata, "ensemble_frames.pkl")

## Naive Temporal

In [72]:
y = annotations["accident_frame"].values
y_total_frames, y_durations_sec = annotations["no_frames"].values, annotations["duration"].values

y_fps = y_total_frames / y_durations_sec

x_naive = np.array(y_total_frames / 2).round()
print("Naive Temporal - Middle Frame")
for weight in [1/2, 1, 2]:
    sigmas = y_fps * weight
    acc = temporal_accuracy(x_naive, y, sigmas)
    print(f" - Weight, Accuracy: {weight:.2f}, {acc:.3f}")


# AVG_ACCIDENT_TIME = 4.9
AVG_ACCIDENT_TIME = np.mean(annotations["accident_time"])
x_naive = np.array(y_fps * AVG_ACCIDENT_TIME).round()

print("Naive Temporal - Average accident time")
for weight in [1/2, 1, 2]:
    sigmas = y_fps * weight
    acc = temporal_accuracy(x_naive, y, sigmas)
    print(f" - Weight, Accuracy: {weight:.2f}, {acc:.3f}")


Naive Temporal - Middle Frame
 - Weight, Accuracy: 0.50, 0.108
 - Weight, Accuracy: 1.00, 0.189
 - Weight, Accuracy: 2.00, 0.295
Naive Temporal - Average accident time
 - Weight, Accuracy: 0.50, 0.158
 - Weight, Accuracy: 1.00, 0.300
 - Weight, Accuracy: 2.00, 0.541


## Spatial localization

In [73]:
import math
from itertools import combinations


def bbox_center(x1: float, y1: float, x2: float, y2: float) -> tuple[float, float]:
    return x1 + (abs(x2 - x1) / 2), y1 + (abs(y2 - y1) / 2)

def euclidean_distance(p1, p2):
    return math.sqrt((p1[0] - p2[0])**2 + (p1[1] - p2[1])**2)

def midpoint(p1, p2):
    return ((p1[0] + p2[0]) / 2, (p1[1] + p2[1]) / 2)

def normalize_point(x: float, y: float, width: int, height: int) -> tuple[float, float]:
    return x / width, y / height



x_points, y_points = [], []
for i, row in tqdm(annotations.iterrows(), total=len(annotations)):
    accident_frame = row["accident_frame"]
    accident_x0, accident_y0, accident_x1, accident_y1 = row["x1"], row["y1"], row["x2"], row["y2"]
    accident_center = (row["center_x"], row["center_y"])
    # accident_center = bbox_center(accident_x0, accident_y0, accident_x1, accident_y1)
    y_points.append(accident_center)

    detections_path = row["detections_path"]
    detections = load_json(detections_path)

    frame_detections = [det for det in detections if int(det["frame_number"]) == int(accident_frame)]
    if len(frame_detections) == 0:  # Default to frame center
        x_points.append((0.5, 0.5))
        continue

    frame_detections = frame_detections[0]

    if "bboxes" not in frame_detections:
        x_points.append((0.5, 0.5))  # Default to frame center
        continue

    cls_labels = frame_detections["labels"]
    if len(ALLOWED_CLASSES) > 0:  # filter allowed classes if defined
        filtered_indexes = [i for i, label in enumerate(cls_labels) if label in ALLOWED_CLASSES]
    else:
        filtered_indexes = list(range(len(cls_labels)))

    bboxes = frame_detections["bboxes"]
    bbox_centers = [bbox_center(x1, y1, x2, y2) for (x1, y1, x2, y2) in frame_detections["bboxes"]]

    if len(bbox_centers) < 1:
        x_points.append((0.5, 0.5))  # Default to frame center
        continue
    elif len(bbox_centers) < 2:
        x_points.append(normalize_point(x=bbox_centers[0][0], y=bbox_centers[0][1], width=row["width"], height=row["height"]))
        continue

    (c1, c2) = min(
        combinations(bbox_centers, 2),
        key=lambda pair: euclidean_distance(pair[0], pair[1])
    )

    # prediction = midpoint of those two centers
    prediction_center = midpoint(c1, c2)
    x_points.append(normalize_point(x=prediction_center[0], y=prediction_center[1], width=row["width"], height=row["height"]))


x_heuristic_points = np.array(x_points)
y_points = np.array(y_points)

100%|██████████| 2027/2027 [00:16<00:00, 123.00it/s]


In [ ]:
def spatial_accuracy(x: list[tuple[float, float]], y: list[tuple[float, float]], sigmas: list[tuple[float, float]]) -> float:
    x, y = np.array(x), np.array(y)
    sigmas = np.array(sigmas)

    scores = np.exp(-(
        ((x[:, 0] - y[:, 0])**2 / (2 * sigmas[:, 0]**2)) +
        ((x[:, 1] - y[:, 1])**2 / (2 * sigmas[:, 1]**2))
    ))
    score = np.mean(scores)
    return float(score)


anns_width = (annotations["x2"] - annotations["x1"]).abs()
anns_height = (annotations["y2"] - annotations["y1"]).abs()

x_norm = x_heuristic_points
y_norm = y_points

sigmas = np.stack([
      anns_width,
      anns_height,
], axis=1)
sigmas = np.mean(sigmas, axis=0)
print(sigmas)
sigmas = np.ones(y_norm.shape) * sigmas

print("Nearest Bboxes Spatial")
for weight in [1/2, 1, 2]:
    # acc = spatial_accuracy(x_points, y_points, sigmas * weight)
    acc = spatial_accuracy(x_norm, y_norm, sigmas * weight)
    print(f" - Weight, Accuracy: {weight:.2f}, {acc:.3f}")



## Naive spatial

In [ ]:

y_norm = annotations[["center_x", "center_y"]].to_numpy()
x_norm = np.ones_like(y_points) * 0.5


sigmas = np.stack([
    anns_width,
    anns_height,
], axis=1)
sigmas = np.mean(sigmas, axis=0)
sigmas = np.ones(y_norm.shape) * sigmas

print("Naive Spatial")
for weight in [1/2, 1, 2]:
    # acc = spatial_accuracy(x_points, y_points, sigmas * weight)
    acc = spatial_accuracy(x_norm, y_norm, sigmas * weight)
    print(f" - Weight, Accuracy: {weight:.2f}, {acc:.3f}")


# End to End Heuristic

In [ ]:
x_points, y_points = [], []
for i, row in tqdm(annotations.iterrows(), total=len(annotations)):
    # accident_frame = row["accident.frame"]
    accident_frame = x_ensemble[i]
    accident_center = (row["center_x"], row["center_y"])
    y_points.append(accident_center)

    detections_path = row["detections_path"]
    detections = load_json(detections_path)

    frame_detections = [det for det in detections if int(det["frame_number"]) == int(accident_frame)]
    if len(frame_detections) == 0:  # Default to frame center
        x_points.append((0.5, 0.5))
        continue

    frame_detections = frame_detections[0]

    if "bboxes" not in frame_detections:
        x_points.append((0.5, 0.5))  # Default to frame center
        continue

    cls_labels = frame_detections["labels"]
    if len(ALLOWED_CLASSES) > 0:  # filter allowed classes if defined
        filtered_indexes = [i for i, label in enumerate(cls_labels) if label in ALLOWED_CLASSES]
    else:
        filtered_indexes = list(range(len(cls_labels)))

    bboxes = frame_detections["bboxes"]
    bbox_centers = [bbox_center(x1, y1, x2, y2) for (x1, y1, x2, y2) in frame_detections["bboxes"]]

    if len(bbox_centers) < 1:
        x_points.append((0.5, 0.5))  # Default to frame center
        continue
    elif len(bbox_centers) < 2:
        x_points.append(normalize_point(x=bbox_centers[0][0], y=bbox_centers[0][1], width=row["width"], height=row["height"]))
        continue

    (c1, c2) = min(
        combinations(bbox_centers, 2),
        key=lambda pair: euclidean_distance(pair[0], pair[1])
    )

    # prediction = midpoint of those two centers
    prediction_center = midpoint(c1, c2)
    x_points.append(normalize_point(x=prediction_center[0], y=prediction_center[1], width=row["width"], height=row["height"]))



x_heuristic_points = np.array(x_points)
y_points = np.array(y_points)

In [ ]:
x_norm = x_heuristic_points
y_norm = y_points

sigmas = np.stack([
    anns_width,
    anns_height,
], axis=1)
sigmas = np.mean(sigmas, axis=0)
print(sigmas)
sigmas = np.ones(y_norm.shape) * sigmas

print("End-to-End Bboxes Spatial")
for weight in [1/2, 1, 2]:
    # acc = spatial_accuracy(x_points, y_points, sigmas * weight)
    acc = spatial_accuracy(x_norm, y_norm, sigmas * weight)
    print(f" - Weight, Accuracy: {weight:.2f}, {acc:.3f}")